Create Trades dataset

In [10]:
import pandas as pd
import numpy as np

# -----------------------------
# Configuration
# -----------------------------
np.random.seed(42)

N = 10000  # number of trades (change as needed)

NEGATIVE_RATE = 0.025   # ~2.5% negatives
NULL_RATE = 0.015       # ~1.5% nulls


# Generate base data
trade_ids = np.arange(1, N + 1)

security_ids = np.random.choice(
    [f"SEC_{i:04d}" for i in range(1, 501)],
    size=N
)

prices = np.round(np.random.lognormal(mean=4, sigma=0.3, size=N), 2)
quantities = np.random.randint(1, 10000, size=N)

timestamps = pd.to_datetime("2025-01-01") + pd.to_timedelta(
    np.random.randint(0, 60 * 60 * 24 * 180, size=N),
    unit="s"
)

df = pd.DataFrame({
    "TradeID": trade_ids,
    "SecurityID": security_ids,
    "Price": prices,
    "Quantity": quantities,
    "Timestamp": timestamps
})


# put negative values in trades df
num_negative = int(N * NEGATIVE_RATE)

neg_price_idx = np.random.choice(df.index, num_negative, replace=False)
neg_qty_idx = np.random.choice(df.index, num_negative, replace=False)

df.loc[neg_price_idx, "Price"] *= -1
df.loc[neg_qty_idx, "Quantity"] *= -1

# put null values in trades df
num_nulls = int(N * NULL_RATE)

for col in ["Price", "Quantity"]:
    null_idx = np.random.choice(df.index, num_nulls, replace=False)
    df.loc[null_idx, col] = np.nan


# Result
print(df.head())
print("\nNull percentages:")
print(df.isnull().mean() * 100)


   TradeID SecurityID  Price  Quantity           Timestamp
0        1   SEC_0103  51.69    5255.0 2025-01-05 17:36:18
1        2   SEC_0436  47.90    8949.0 2025-01-08 19:18:10
2        3   SEC_0349  55.98    9247.0 2025-06-09 21:59:33
3        4   SEC_0271  34.60    4355.0 2025-04-05 20:50:55
4        5   SEC_0107  59.08    3454.0 2025-06-15 22:33:42

Null percentages:
TradeID       0.0
SecurityID    0.0
Price         1.5
Quantity      1.5
Timestamp     0.0
dtype: float64


In [11]:
df.head()

,TradeID,SecurityID,Price,Quantity,Timestamp
0,1,SEC_0103,51.69,5255.0,2025-01-05 17:36:18
1,2,SEC_0436,47.90,8949.0,2025-01-08 19:18:10
2,3,SEC_0349,55.98,9247.0,2025-06-09 21:59:33
3,4,SEC_0271,34.60,4355.0,2025-04-05 20:50:55
4,5,SEC_0107,59.08,3454.0,2025-06-15 22:33:42


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   TradeID     10000 non-null  int64         
 1   SecurityID  10000 non-null  object        
 2   Price       9850 non-null   float64       
 3   Quantity    9850 non-null   float64       
 4   Timestamp   10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 390.8+ KB


Create Securities dataset

In [13]:
security_types = np.random.choice(
    [
        "Equity",
        "Fixed Income",
        "FX",
        "Derivative",
        "ETF"
    ],
    size=10000,
    p=[0.45, 0.15, 0.05, 0.2, 0.15]  # realistic distribution
)

In [14]:
sec_df = pd.DataFrame({
    "SecurityID" : df['SecurityID'],
    "SecurityType" : security_types
})

In [15]:
sec_df.head()

,SecurityID,SecurityType
0,SEC_0103,Equity
1,SEC_0436,Fixed Income
2,SEC_0349,Equity
3,SEC_0271,Derivative
4,SEC_0107,Equity


In [16]:
sec_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   SecurityID    10000 non-null  object
 1   SecurityType  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [28]:
# saving the results to csv for analysis in Tableau
df.to_csv('C:/Users/Nishkarsh Khokhar/Desktop/trades.csv')
sec_df.to_csv('C:/Users/Nishkarsh Khokhar/Desktop/securities.csv')

Loading these two files into sqlite

Note: Created these two files as csv separately for Tableau, but will load the created dataframes directly in the database

In [17]:
import sqlite3

In [18]:
# checking where the database is created
import os
print(os.getcwd())

c:\Users\Nishkarsh Khokhar\Desktop


In [19]:
os.chdir('C:/Users/Nishkarsh Khokhar/Desktop/')

In [20]:
conn = sqlite3.connect("trading.db")

In [21]:
cursor = conn.cursor()

In [22]:

sec_df.to_sql(
    "Securities",
    conn,
    if_exists="replace",
    index=False
)

df.to_sql(
    "Trades",
    conn,
    if_exists="replace", 
    index=False)


10000

Checking if databases loaded correctly and are present

In [23]:
query = """
SELECT
    t.TradeID,
    t.Price,
    t.Quantity,
    s.SecurityType
FROM Trades t
JOIN Securities s
  ON t.SecurityID = s.SecurityID
LIMIT 5
"""

result = pd.read_sql(query, conn)
print(result)


   TradeID  Price  Quantity SecurityType
0        1  51.69    5255.0   Derivative
1        1  51.69    5255.0   Derivative
2        1  51.69    5255.0          ETF
3        1  51.69    5255.0          ETF
4        1  51.69    5255.0          ETF


In [24]:
query = """
SELECT
    *
FROM Trades t
LIMIT 5
"""

result = pd.read_sql(query, conn)
print(result)


   TradeID SecurityID  Price  Quantity            Timestamp
0        1   SEC_0103  51.69    5255.0  2025-01-05 17:36:18
1        2   SEC_0436  47.90    8949.0  2025-01-08 19:18:10
2        3   SEC_0349  55.98    9247.0  2025-06-09 21:59:33
3        4   SEC_0271  34.60    4355.0  2025-04-05 20:50:55
4        5   SEC_0107  59.08    3454.0  2025-06-15 22:33:42


In [25]:
result.head()

,TradeID,SecurityID,Price,Quantity,Timestamp
0,1,SEC_0103,51.69,5255.0,2025-01-05 17:36:18
1,2,SEC_0436,47.90,8949.0,2025-01-08 19:18:10
2,3,SEC_0349,55.98,9247.0,2025-06-09 21:59:33
3,4,SEC_0271,34.60,4355.0,2025-04-05 20:50:55
4,5,SEC_0107,59.08,3454.0,2025-06-15 22:33:42


In [27]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,sqlite_sequence
1,exceptions
2,Securities
3,Trades
4,DataQualityExceptions


In [71]:
pd.read_sql(
    "SELECT * FROM exceptions ORDER BY DetectedAt DESC",
    conn
)

,TradeID,RuleViolated,Reason,DetectedAt
0,507,Timeliness,Trade timestamp older than allowed window,2025-12-31T23:15:43.952459
1,530,Timeliness,Trade timestamp older than allowed window,2025-12-31T23:15:43.952459
2,690,Timeliness,Trade timestamp older than allowed window,2025-12-31T23:15:43.952459
3,780,Timeliness,Trade timestamp older than allowed window,2025-12-31T23:15:43.952459
4,827,Timeliness,Trade timestamp older than allowed window,2025-12-31T23:15:43.952459
...,...,...,...,...
176,690,Timeliness,Trade timestamp older than allowed window,2025-12-31T20:43:04.125359
177,780,Timeliness,Trade timestamp older than allowed window,2025-12-31T20:43:04.125359
178,827,Timeliness,Trade timestamp older than allowed window,2025-12-31T20:43:04.125359
179,891,Timeliness,Trade timestamp older than allowed window,2025-12-31T20:43:04.125359
